# Phase 5 — Évaluation du pipeline YouTube Quality Analyzer

Ce notebook agrège tous les résultats d'évaluation (H1–H3) et produit les visualisations finales.

**Hypothèses :**
- H1 : Pearson r(Score_Global, gold_score) >= 0.60
- H2 : DeltaPearson(multi-agents − LLM unique) > 0.10
- H3 : contribution A4 > A3 > A5 (ablation)

In [ ]:
import sys, json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

RESULTS_DIR = ROOT / 'evaluation' / 'results'
GOLD_PATH   = ROOT / 'data' / 'gold_standard' / 'gold_standard.jsonl'
PREDS_PATH  = ROOT / 'data' / 'predictions' / 'pipeline_predictions.jsonl'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print('Configuration OK — ROOT:', ROOT)

## 1. Chargement des résultats JSON

In [ ]:
def load_result(filename):
    path = RESULTS_DIR / filename
    if not path.exists():
        print(f'  [MANQUANT] {filename} — lancez le script correspondant')
        return {}
    with open(path, encoding='utf-8') as f:
        return json.load(f)

metrics      = load_result('metrics_report.json')
baseline_cmp = load_result('baseline_comparison.json')
ablation     = load_result('ablation_study.json')
errors       = load_result('error_analysis.json')

print('Fichiers charges :')
for name, data in [('metrics_report', metrics), ('baseline_comparison', baseline_cmp),
                   ('ablation_study', ablation), ('error_analysis', errors)]:
    status = f'{len(data)} cles' if data else 'ABSENT'
    print(f'  {name:<25} {status}')

## 2. H1 — Corrélation Pearson (Score_Global vs gold_score)

In [ ]:
if metrics:
    r    = metrics.get('pearson_r', float('nan'))
    mae  = metrics.get('mae_score_global', float('nan'))
    rmse = metrics.get('rmse_score_global', float('nan'))
    h1   = metrics.get('h1_satisfied', False)
    n    = metrics.get('n_evaluated', '?')
    print('=' * 50)
    print('H1 — Correlation Score_Global vs Gold')
    print('=' * 50)
    print(f'  n evalues  : {n}')
    ok_str = 'H1 validee' if h1 else 'H1 non validee'
    print(f'  Pearson r  : {r:.4f}  {ok_str} (seuil >= 0.60)')
    print(f'  MAE        : {mae:.2f}')
    print(f'  RMSE       : {rmse:.2f}')
else:
    print('Resultats H1 non disponibles. Lancez compute_metrics.py.')

In [ ]:
from evaluation.compute_metrics import load_gold, load_predictions

if GOLD_PATH.exists() and PREDS_PATH.exists():
    gold   = load_gold(str(GOLD_PATH))
    preds  = load_predictions(str(PREDS_PATH))
    merged = gold.merge(preds, on='comment_id', suffixes=('_gold', '_pred'))
    if 'gold_score' in merged.columns and 'score_global' in merged.columns:
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.scatter(merged['gold_score'], merged['score_global'],
                   alpha=0.6, s=40, color='steelblue', edgecolors='white', linewidths=0.5)
        ax.plot([0, 100], [0, 100], 'r--', lw=1, label='y = x')
        ax.set_xlabel('Gold score [0-100]')
        ax.set_ylabel('Score_Global predit [0-100]')
        r_val = metrics.get('pearson_r', '?')
        ax.set_title(f'H1 — Pearson r = {r_val}')
        ax.legend()
        plt.tight_layout()
        plt.savefig(RESULTS_DIR / 'h1_scatter.png')
        plt.show()
else:
    print('Gold standard ou predictions manquants — scatter non disponible.')

## 3. H2 — Pipeline multi-agents vs LLM unique (baseline)

In [ ]:
if baseline_cmp:
    bl = baseline_cmp.get('baseline_llm_single', {})
    pl = baseline_cmp.get('pipeline_multi_agent', {})
    dr = baseline_cmp.get('delta_pearson_r', float('nan'))
    h2 = baseline_cmp.get('h2_satisfied', False)
    print('=' * 56)
    print('H2 — Comparaison baseline vs pipeline')
    print('=' * 56)
    print(f'{"Methode":<30} {"Pearson r":>10} {"MAE":>8}')
    print('-' * 56)
    r_bl  = bl.get('pearson_r', float('nan'))
    m_bl  = bl.get('mae', float('nan'))
    r_pl  = pl.get('pearson_r', float('nan'))
    m_pl  = pl.get('mae', float('nan'))
    print(f'{"LLM unique (baseline)":<30} {r_bl:>10.4f} {m_bl:>8.2f}')
    print(f'{"Pipeline multi-agents":<30} {r_pl:>10.4f} {m_pl:>8.2f}')
    print('-' * 56)
    ok_str = 'H2 validee (D > 0.10)' if h2 else 'H2 non validee (D <= 0.10)'
    print(f'DeltaPearson r = {dr:+.4f}  {ok_str}')
else:
    print('Resultats H2 non disponibles. Lancez baseline_comparison.py.')

In [ ]:
if baseline_cmp:
    bl = baseline_cmp.get('baseline_llm_single', {})
    pl = baseline_cmp.get('pipeline_multi_agent', {})
    methods = ['LLM unique\n(baseline)', 'Pipeline\nmulti-agents']
    r_vals  = [bl.get('pearson_r', 0), pl.get('pearson_r', 0)]
    fig, ax = plt.subplots(figsize=(5, 4))
    bars = ax.bar(methods, r_vals, color=['#e07b54', '#4a90d9'], width=0.5, edgecolor='white')
    ax.axhline(0.60, color='green', linestyle='--', lw=1.5, label='Seuil H1 (r=0.60)')
    ax.set_ylim(0, 1.0)
    ax.set_ylabel('Pearson r')
    ax.set_title('H2 — Comparaison baseline vs pipeline')
    for bar, val in zip(bars, r_vals):
        ax.text(bar.get_x() + bar.get_width() / 2, val + 0.02, f'{val:.3f}', ha='center', fontsize=10)
    ax.legend()
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'h2_comparison.png')
    plt.show()

## 4. H3 — Étude d'ablation (contribution par agent)

In [ ]:
if ablation:
    full = ablation.get('full_pipeline', {})
    abl  = ablation.get('ablation', {})
    rank = ablation.get('ranking_by_contribution', [])
    h3   = ablation.get('h3_satisfied', False)
    print('=' * 60)
    print('H3 — Etude d ablation — Contribution par agent')
    print('=' * 60)
    r_full = full.get('pearson_r', float('nan'))
    print(f'{"Pipeline":<30} {"Pearson r":>10} {"Delta":>10}')
    print('-' * 60)
    print(f'{"Complet (A1-A7)":<30} {r_full:>10.4f} {"(reference)":>10}')
    for agent, res in abl.items():
        r_no  = res.get('pearson_r_without', float('nan'))
        delta = res.get('delta_pearson_r', float('nan'))
        print(f'  {"Sans " + agent:<28} {r_no:>10.4f} {delta:>+10.4f}')
    print('-' * 60)
    ranked_agents = ', '.join(r['agent'] for r in rank)
    print(f'Classement : {ranked_agents}')
    ok_str = 'validee' if h3 else 'non validee'
    print(f'H3 : A4 > A3 > A5 — {ok_str}')
else:
    print('Resultats H3 non disponibles. Lancez ablation_study.py.')

In [ ]:
if ablation:
    full_r = ablation.get('full_pipeline', {}).get('pearson_r', 0)
    abl    = ablation.get('ablation', {})
    agents = list(abl.keys())
    deltas = [abl[a].get('delta_pearson_r', 0) for a in agents]
    colors = ['#e07b54' if d > 0 else '#aaa' for d in deltas]
    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.bar([f'Sans {a}' for a in agents], deltas, color=colors, width=0.5)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_ylabel('Delta Pearson r (contribution)')
    ax.set_title(f'H3 — Ablation (Pipeline complet : r={full_r:.3f})')
    for bar, val in zip(bars, deltas):
        offset = 0.002 if val >= 0 else -0.008
        ax.text(bar.get_x() + bar.get_width() / 2, val + offset,
                f'{val:+.4f}', ha='center', fontsize=9)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'h3_ablation.png')
    plt.show()

## 5. Métriques sentiment et bruit

In [ ]:
if metrics:
    acc = metrics.get('sentiment_accuracy')
    f1  = metrics.get('sentiment_f1_macro')
    kap = metrics.get('sentiment_kappa')
    if acc is not None:
        print('-- Sentiment (3 classes) --')
        print(f'  Accuracy  : {acc:.4f}')
        print(f'  F1-macro  : {f1:.4f}')
        kap_str = 'kappa > 0.70' if kap and kap > 0.70 else 'kappa <= 0.70'
        print(f'  Kappa     : {kap:.4f}  {kap_str}')
    else:
        print('Metriques sentiment non disponibles.')
if errors:
    noise = errors.get('noise_detection', {})
    if noise.get('available'):
        print('\n-- Bruit (A5) --')
        print(f'  Precision : {noise["precision"]:.3f}')
        print(f'  Rappel    : {noise["recall"]:.3f}')
        print(f'  F1        : {noise["f1"]:.3f}')

## 6. Analyse des outliers de score

In [ ]:
if errors:
    n_out = errors.get('n_outliers', 0)
    rate  = errors.get('outlier_rate', 0)
    thr   = errors.get('outlier_threshold', 20)
    items = errors.get('score_outliers', [])
    print(f'Outliers (|err| > {thr}) : {n_out} ({rate:.1%})')
    if items:
        df_out = pd.DataFrame(items[:10])
        display(df_out[['comment_id', 'gold_score', 'pred_score', 'abs_error', 'direction']])
else:
    print('Analyse erreurs non disponible. Lancez error_analysis.py.')

## 7. Récapitulatif final

In [ ]:
print('=' * 65)
print('RECAPITULATIF — Validation des hypotheses')
print('=' * 65)

def _status(val, threshold, direction='>='):
    if val is None:
        return 'N/A'
    ok = (val >= threshold) if direction == '>=' else (val > threshold)
    return f'OK  {val:.4f}' if ok else f'KO  {val:.4f}'

h1_r  = metrics.get('pearson_r') if metrics else None
h2_dr = baseline_cmp.get('delta_pearson_r') if baseline_cmp else None
h3_ok = ablation.get('h3_satisfied') if ablation else None

print(f'H1 — Pearson r >= 0.60       : {_status(h1_r, 0.60)}')
print(f'H2 — DeltaPearson > 0.10     : {_status(h2_dr, 0.10, ">")}')
h3_str = 'OK validee' if h3_ok else ('KO non validee' if h3_ok is not None else 'N/A')
print(f'H3 — Contribution A4>A3>A5   : {h3_str}')
print('=' * 65)